In [ ]:
import math
import jaydebeapi
import psycopg2
import json
import os
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime, timezone
from sklearn.linear_model import LogisticRegression

In [ ]:
USE_H2_DB = os.getenv('USE_H2_DB')
DB_PW = os.getenv('DB_PASSWORD_LLMAPE')
DB_HOST = os.getenv('DB_HOST')

if DB_HOST == None:
	DB_HOST = 'localhost'

Establish connection with the correct database

In [ ]:
conn = None

if USE_H2_DB == "true":

	h2_jar_path = "./drivers/h2.jar"
	jdbc_url = "jdbc:h2:file:../../../../target/h2db/db/llmape;DB_CLOSE_DELAY=-1"
	driver_class = "org.h2.Driver"

	conn = jaydebeapi.connect(
		driver_class,
		jdbc_url,
		["llmape", DB_PW],
		h2_jar_path
	)

else:
	conn = psycopg2.connect(
    host=DB_HOST,
    port=5432,
    database="llmape",
    user="llmape",
    password=DB_PW
)

Set current category (use category you want to use for the pipeline)

In [ ]:
CATEGORY = 'OVERALL' # 'OVERALL' 'HINT_GENERATION' 'EXERCISE_GENERATION' 'CODE_ASSESSMENT'

category_figures_text = "across all categories"

if CATEGORY != 'OVERALL':
	category_figures_text = "in category " + CATEGORY.replace("_", " ").lower() 

category_figures_text

Set current cutoff date (use data you would like to use as cutoff - later data will not be fetched for most queries)

In [ ]:
CUTOFF_DATE = '2025-12-01'

date_obj = datetime.strptime(CUTOFF_DATE, "%Y-%m-%d")
european_format = date_obj.strftime("%d.%m.%Y")
category_and_cutoff_date_figures_text = category_figures_text + " until " + european_format

category_and_cutoff_date_figures_text

Determine if only data for currently active models should be used (only affects data regarding battles - prompt only data is not affected)

In [ ]:
USE_ACTIVE_ONLY = True

Retrieve the model names and their ids for later queries and lookups

In [ ]:
query = f"""
	SELECT ID, MODEL_NAME FROM MODEL WHERE (Active = TRUE OR {USE_ACTIVE_ONLY} = FALSE)
"""

models_df = pd.read_sql(query, conn)
models_df.columns = [col.upper() for col in models_df.columns]

all_model_ids = models_df['ID'].to_list()
all_model_names = models_df['MODEL_NAME'].to_list()

all_model_names

In [ ]:
models_df

In [ ]:
model_id_name_mapping = dict(zip(models_df['ID'], models_df['MODEL_NAME']))
model_id_name_mapping

To translate ids to model names using the dictionary, we use:

Rename columns:
df = df.rename(columns=mapping)

Rename index (first column cells):
df = df.rename(index=mapping)

Rename each cell:
df = df.applymap(mapping.get)

Retrieve prompts and battles data from the database

In [ ]:
query = f"""
	SELECT * 
	FROM BATTLE b 
	JOIN PROMPT p ON b.PROMPT_ID = p.ID
	JOIN MODEL m1 ON b.model1_id = m1.id
	JOIN MODEL m2 ON b.model2_id = m2.id
	WHERE VOTE_TIMESTAMP IS NOT NULL
	AND (p.category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	AND (VOTE_TIMESTAMP < '{CUTOFF_DATE}')
	AND (m1.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
	AND (m2.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
"""

battles_and_prompts_df = pd.read_sql(query, conn)
battles_and_prompts_df.columns = [col.upper() for col in battles_and_prompts_df.columns]

battles_and_prompts_df

In [ ]:
def label_winner(row):
	if pd.isna(row["WINNER_MODEL_ID"]):
		return "Tie"
	elif row["WINNER_MODEL_ID"] == row ["MODEL1_ID"]:
		return "Model A"
	elif row["WINNER_MODEL_ID"] == row ["MODEL2_ID"]:
		return "Model B"
	else:
		return "Unknown"

In [ ]:
if CATEGORY == 'OVERALL':
	fig = px.bar(battles_and_prompts_df["CATEGORY"].value_counts(), title="Number of voted battles " + category_and_cutoff_date_figures_text, text_auto=True, height=400)

	fig.update_layout(xaxis_title="Category", yaxis_title="Number of voted battles", showlegend=False, font_size=16)

	fig.show()

In [ ]:
battles_and_prompts_df["WINNER"] = battles_and_prompts_df.apply(label_winner, axis=1)

In [ ]:
battles_and_prompts_df

In [ ]:
fig = px.bar(battles_and_prompts_df["WINNER"].value_counts(), title="Counts of wins and ties " + category_and_cutoff_date_figures_text, text_auto=True, height=400)

fig.update_layout(xaxis_title="Battle Outcome", yaxis_title="Count of wins", showlegend=False, font_size=16)

fig

In [ ]:
modelBattlesSeries = pd.concat([battles_and_prompts_df["MODEL1_ID"], battles_and_prompts_df["MODEL2_ID"]])
modelBattlesSeries = modelBattlesSeries.rename("MODEL_ID") 
modelBattlesDf = modelBattlesSeries.to_frame()
modelBattlesIDAndNameDf = modelBattlesDf.merge(models_df, left_on="MODEL_ID", right_on="ID", how="left")
modelBattlesIDAndNameDf

In [ ]:
fig = px.bar(modelBattlesIDAndNameDf["MODEL_NAME"].value_counts(), title="Battle count for each model " + category_and_cutoff_date_figures_text, text_auto=True, height=400)

fig.update_layout(xaxis_title="Model name", yaxis_title="Battle count", showlegend=False, font_size=16)

fig

Retrieve battle pairings and their win counts used for win matrix

In [ ]:
query = f"""
	SELECT
		winner.id AS winner_model,
		loser.id AS loser_model,
		COUNT(*) AS win_count
	FROM battle
	JOIN model AS winner ON battle.winner_model_id = winner.id
	JOIN model AS model1 ON battle.model1_id = model1.id
	JOIN model AS model2 ON battle.model2_id = model2.id
	-- Determine which of model1 or model2 is the loser
	JOIN model AS loser ON (
		(battle.winner_model_id = model1.id AND loser.id = model2.id) OR
		(battle.winner_model_id = model2.id AND loser.id = model1.id)
	)
	JOIN prompt ON battle.prompt_id = prompt.id
	WHERE (prompt.category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	AND (battle.VOTE_TIMESTAMP < '{CUTOFF_DATE}')
	AND (model1.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
	AND (model2.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
	GROUP BY winner.id, loser.id
	ORDER BY win_count DESC;
"""
df = pd.read_sql(query, conn)
df.columns = ['winner_model', 'loser_model', 'win_count']

df

In [ ]:
win_matrix_df = df.pivot_table(
    index="winner_model",  # Rows
    columns="loser_model",  # Columns
    values="win_count",
    fill_value=0  # Replace NaN with 0 (i.e., no wins)
)

#ensure all models appear on both axes
win_matrix_df = win_matrix_df.reindex(index=all_model_ids, columns=all_model_ids, fill_value=0)
win_matrix_df = win_matrix_df.rename(columns=model_id_name_mapping, index=model_id_name_mapping)

win_matrix_df

Retrieve battle pairings with their tie counts for tie matrix

In [ ]:
query = f"""
	SELECT
		model1.id AS modelA,
		model2.id AS modelB,
		COUNT(*) AS tie_count
	FROM battle
	JOIN model AS model1 ON battle.model1_id = model1.id
	JOIN model AS model2 ON battle.model2_id = model2.id
	JOIN prompt ON battle.prompt_id = prompt.id
	WHERE
		(battle.winner_model_id IS NULL)
		AND (battle.vote_timestamp IS NOT NULL)
		AND (battle.vote_timestamp < '{CUTOFF_DATE}')
		AND (prompt.category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
		AND (model1.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
		AND (model2.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
	GROUP BY model1.id, model2.id
"""
df_ties = pd.read_sql(query, conn)
df_ties.columns = ['modelA', 'modelB', 'tie_count']

df_ties

In [ ]:
tie_matrix_df = df_ties.pivot_table(
    index="modelA",  # Rows
    columns="modelB",  # Columns
    values="tie_count",
    fill_value=0  # Replace NaN with 0 (i.e., no wins)
)

#ensure all models appear on both axes
tie_matrix_df = tie_matrix_df.reindex(index=all_model_ids, columns=all_model_ids, fill_value=0)
tie_matrix_df = tie_matrix_df.rename(columns=model_id_name_mapping, index=model_id_name_mapping)

tie_matrix_df = tie_matrix_df + tie_matrix_df.T

tie_matrix_df

Calculate amount of battles between each model (battle matrix)

In [ ]:
battle_matrix_df = win_matrix_df + win_matrix_df.T + tie_matrix_df
battle_matrix_df

In [ ]:
fig = px.imshow(battle_matrix_df, title="Battles between Model A and Model B " + category_and_cutoff_date_figures_text,
				labels=dict(x="Model A", y="Model B", color="Battle count"),
				text_auto=True)

fig.update_xaxes(side="top")
fig.update_layout(title=dict(
    y=0.05,               # 0 = bottom, 1 = top
    x=0.5,             # center the title horizontally
))
fig.show()

Calculate battle matrix without ties

In [ ]:
battle_matrix_without_ties_df = win_matrix_df + win_matrix_df.T

fig = px.imshow(battle_matrix_without_ties_df, title="Battles between Model A and Model B without ties " + category_and_cutoff_date_figures_text,
				labels=dict(x="Model A", y="Model B", color="Battle count"),
				text_auto=True)

fig.update_xaxes(side="top")
fig.update_layout(title=dict(
    y=0.05,               # 0 = bottom, 1 = top
    x=0.5,             # center the title horizontally
))
fig.show()

In [ ]:
fig = px.imshow(win_matrix_df, title="Wins of Model A against Model B " + category_and_cutoff_date_figures_text,
				labels=dict(x="Model B", y="Model A", color="Win count"),
				text_auto=True)

fig.update_xaxes(side="top")
fig.update_layout(title=dict(
    y=0.05,               # 0 = bottom, 1 = top
    x=0.5,             # center the title horizontally
))
fig.show()

In [ ]:
winning_percentage_matrix_without_ties_df = (win_matrix_df / battle_matrix_without_ties_df).fillna(0).replace([np.inf, -np.inf], 0).round(2)

fig = px.imshow(winning_percentage_matrix_without_ties_df, title="Fraction of wins of Model A against Model B " + category_and_cutoff_date_figures_text,
				labels=dict(x="Model B", y="Model A", color="Win fraction"),
				text_auto=True)

fig.update_xaxes(side="top")
fig.update_layout(title=dict(
    y=0.05,               # 0 = bottom, 1 = top
    x=0.5,             # center the title horizontally
))
fig.show()

In [ ]:
query = f"""
	SELECT
    SUM(
        CASE 
            WHEN b.winner_model_id = b.model1_id 
                 AND LENGTH(b.model_1_answer) > LENGTH(b.model_2_answer)
              THEN 1
            WHEN b.winner_model_id = b.model2_id
                 AND LENGTH(b.model_2_answer) > LENGTH(b.model_1_answer)
              THEN 1
            ELSE 0
        END
    ) AS longer_answer_wins,

    SUM(
        CASE 
            WHEN b.winner_model_id = b.model1_id 
                 AND LENGTH(b.model_1_answer) < LENGTH(b.model_2_answer)
              THEN 1
            WHEN b.winner_model_id = b.model2_id
                 AND LENGTH(b.model_2_answer) < LENGTH(b.model_1_answer)
              THEN 1
            ELSE 0
        END
    ) AS shorter_answer_wins

	FROM battle b
	JOIN PROMPT p ON b.PROMPT_ID = p.ID
	JOIN model AS model1 ON b.model1_id = model1.id
	JOIN model AS model2 ON b.model2_id = model2.id 
	WHERE (winner_model_id IS NOT NULL)
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	AND (model1.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
	AND (model2.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
"""

winner_per_response_length_df = pd.read_sql(query, conn)
winner_per_response_length_df.columns = ['Longer answer wins', 'Shorter answer wins']

winner_per_response_length_df

In [ ]:
winner_per_response_length_df_melted = winner_per_response_length_df.melt(var_name="category", value_name="count")
fig = px.bar(winner_per_response_length_df_melted, x="category", y="count", title="Counts of wins and losses with comparison of response lengths " + category_and_cutoff_date_figures_text, text_auto=True, height=400)

fig.update_layout(xaxis_title="Battle Outcomes", yaxis_title="Count of wins", showlegend=False, font_size=16)

fig

Calculate counts of wins and losses with comparison of response lengths (having minimum length difference of LENGTH_DIFFERENCE) characters

In [ ]:
LENGTH_DIFFERENCE = 500 #difference in characters between the prompts to be counted as strongly varying lengths

query = f"""
	SELECT
    SUM(
        CASE 
            WHEN b.winner_model_id = b.model1_id 
                 AND LENGTH(b.model_1_answer) > LENGTH(b.model_2_answer)
              THEN 1
            WHEN b.winner_model_id = b.model2_id
                 AND LENGTH(b.model_2_answer) > LENGTH(b.model_1_answer)
              THEN 1
            ELSE 0
        END
    ) AS longer_answer_wins,

    SUM(
        CASE 
            WHEN b.winner_model_id = b.model1_id 
                 AND LENGTH(b.model_1_answer) < LENGTH(b.model_2_answer)
              THEN 1
            WHEN b.winner_model_id = b.model2_id
                 AND LENGTH(b.model_2_answer) < LENGTH(b.model_1_answer)
              THEN 1
            ELSE 0
        END
    ) AS shorter_answer_wins

	FROM battle b
	JOIN PROMPT p ON b.PROMPT_ID = p.ID
	JOIN model AS model1 ON b.model1_id = model1.id
	JOIN model AS model2 ON b.model2_id = model2.id
	WHERE (winner_model_id IS NOT NULL)
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	AND (ABS(LENGTH(b.model_1_answer) - LENGTH(b.model_2_answer)) > {LENGTH_DIFFERENCE})
	AND (model1.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
	AND (model2.active = TRUE OR ({USE_ACTIVE_ONLY} = FALSE))
"""

winner_per_response_length_with_minimum_length_difference_df = pd.read_sql(query, conn)
winner_per_response_length_with_minimum_length_difference_df.columns = ['Longer answer wins', 'Shorter answer wins']

winner_per_response_length_with_minimum_length_difference_df

In [ ]:
winner_per_response_length_with_minimum_length_difference_df_melted = winner_per_response_length_with_minimum_length_difference_df.melt(var_name="category", value_name="count")
fig = px.bar(winner_per_response_length_with_minimum_length_difference_df_melted, x="category", y="count", title="Counts of wins and losses with comparison of response lengths (having minimum length difference of " + str(LENGTH_DIFFERENCE) + " characters) " + category_and_cutoff_date_figures_text, text_auto=True, height=400)

fig.update_layout(xaxis_title="Battle Outcomes", yaxis_title="Count of wins", showlegend=False, font_size=16)

fig

Calculate number of prompts

In [ ]:
query = f"""
	SELECT COUNT(prompt_text) FROM prompt WHERE timestamp < '{CUTOFF_DATE}' AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
"""
prompts_count_df = pd.read_sql(query, conn)
prompts_count_df.columns = ['Number of prompts']

prompts_count_df

Calculate number of distinct prompts

In [ ]:
query = f"""
	SELECT COUNT(DISTINCT prompt_text) FROM prompt WHERE timestamp < '{CUTOFF_DATE}' AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
"""
distinct_prompts_count_df = pd.read_sql(query, conn)
distinct_prompts_count_df.columns = ['Number of distinct prompts']

distinct_prompts_count_df

Calculate number of distinct rejected prompts

In [ ]:
query = f"""
	SELECT COUNT(DISTINCT prompt_text) FROM prompt WHERE is_rejected = TRUE AND timestamp < '{CUTOFF_DATE}' AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
"""
distinct_rejected_prompts_count_df = pd.read_sql(query, conn)
distinct_rejected_prompts_count_df.columns = ['Number of rejected distinct prompts']

distinct_rejected_prompts_count_df

Calculate number of distinct prompts that were rejected and not rejected at the same time

In [ ]:
query = f"""
	SELECT COUNT(DISTINCT p.prompt_text)
	FROM prompt p
	WHERE (p.is_from_public_page = TRUE OR p.session_id = 'generator-session')
	AND p.timestamp < '{CUTOFF_DATE}'
	AND p.prompt_text IN (
		SELECT p2.prompt_text
		FROM prompt p2
		WHERE (p2.is_from_public_page = TRUE OR p2.session_id = 'generator-session')
		GROUP BY p2.prompt_text
		HAVING SUM(CASE WHEN p2.is_rejected THEN 1 ELSE 0 END) > 0
			AND SUM(CASE WHEN NOT p2.is_rejected THEN 1 ELSE 0 END) > 0
	)
"""
distinct_rejected_and_not_rejected_prompts_count_df = pd.read_sql(query, conn)
distinct_rejected_and_not_rejected_prompts_count_df.columns = ['Number of distinct prompts that are rejected and not rejected at the same time']

distinct_rejected_and_not_rejected_prompts_count_df

Show full data of distinct prompts that were rejected and not rejected at the same time

In [ ]:
query = f"""
	SELECT DISTINCT p1.*
	FROM prompt p1
	WHERE (p1.is_from_public_page = TRUE OR p1.session_id = 'generator-session')
		AND p1.timestamp < '{CUTOFF_DATE}'
		AND EXISTS (
			SELECT 1
			FROM prompt p2
			WHERE p2.prompt_text = p1.prompt_text
				AND (p2.is_from_public_page = TRUE OR p2.session_id = 'generator-session')
				AND p2.is_rejected = TRUE
		)
		AND EXISTS (
			SELECT 1
			FROM prompt p3
			WHERE p3.prompt_text = p1.prompt_text
				AND (p3.is_from_public_page = TRUE OR p3.session_id = 'generator-session')
				AND p3.is_rejected = FALSE
		);
"""
distinct_rejected_and_not_rejected_prompts_df = pd.read_sql(query, conn)

distinct_rejected_and_not_rejected_prompts_df

Distinct prompts that were rejected and not rejected at the same time in the same category

In [ ]:
query = f"""
	SELECT *
		FROM(
		SELECT p.prompt_text, p.category
		FROM prompt p
		WHERE (p.is_from_public_page = TRUE OR p.session_id = 'generator-session')
		AND p.timestamp < '{CUTOFF_DATE}'
		GROUP BY p.prompt_text, p.category
		HAVING SUM(CASE WHEN p.is_rejected THEN 1 ELSE 0 END) > 0
			AND SUM(CASE WHEN NOT p.is_rejected THEN 1 ELSE 0 END) > 0 
	)
"""
distinct_rejected_and_not_rejected_prompts_in_same_category_df = pd.read_sql(query, conn)

distinct_rejected_and_not_rejected_prompts_in_same_category_df

Number of distinct prompts that were rejected and not rejected at the same time in the same category

In [ ]:
query = f"""
	SELECT COUNT(*)
		FROM(
		SELECT p.prompt_text, p.category
		FROM prompt p
		WHERE (p.is_from_public_page = TRUE OR p.session_id = 'generator-session')
		AND p.timestamp < '{CUTOFF_DATE}'
		GROUP BY p.prompt_text, p.category
		HAVING SUM(CASE WHEN p.is_rejected THEN 1 ELSE 0 END) > 0
			AND SUM(CASE WHEN NOT p.is_rejected THEN 1 ELSE 0 END) > 0 
	)
"""
distinct_rejected_and_not_rejected_prompts_in_same_category_count_df = pd.read_sql(query, conn)
distinct_rejected_and_not_rejected_prompts_in_same_category_count_df.columns = ['Number of distinct prompts that are rejected and not rejected at the same time for the same category']


distinct_rejected_and_not_rejected_prompts_in_same_category_count_df

Number of distinct prompts from the non public page

In [ ]:
query = f"""
	SELECT COUNT(DISTINCT prompt_text) 
	FROM prompt 
	WHERE (is_from_public_page = FALSE AND session_id != 'generator-session')
		AND timestamp < '{CUTOFF_DATE}'
"""

distinct_prompts_from_non_public_page_df = pd.read_sql(query, conn)
distinct_prompts_from_non_public_page_df.columns = ['Number of distinct prompts that don\'t come from the public page']


distinct_prompts_from_non_public_page_df

Number of distinct user sessions

In [ ]:
query = f"""
	SELECT COUNT(DISTINCT session_id) FROM prompt WHERE timestamp < '{CUTOFF_DATE}' AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
"""
distinct_user_sessions_df = pd.read_sql(query, conn)
distinct_user_sessions_df.columns = ['Number of user sessions']


distinct_user_sessions_df

Number of distinct prompts per user session from public page or generated

In [ ]:
query = f"""
	SELECT COUNT(DISTINCT prompt_text) AS prompt_count
	FROM prompt
	WHERE (is_from_public_page = TRUE OR session_id = 'generator-session')
	AND timestamp < '{CUTOFF_DATE}'
	GROUP BY session_id
"""
distinct_prompts_per_session_id_public_page_or_generated = pd.read_sql(query, conn)
distinct_prompts_per_session_id_public_page_or_generated.columns = ['Number of distinct prompts per user session for the public page']

distinct_prompts_per_session_id_public_page_or_generated

Maximum number of distinct prompts per user session (public page or generated)

In [ ]:
query = f"""
	SELECT MAX(prompt_count) AS max_distinct_prompts
	FROM (
		SELECT session_id, COUNT(DISTINCT prompt_text) AS prompt_count
		FROM prompt
		WHERE (is_from_public_page = TRUE OR session_id = 'generator-session')
		AND timestamp < '{CUTOFF_DATE}'
		GROUP BY session_id
	) AS counts
"""
max_distinct_prompts_per_session_id_public_page_or_generated = pd.read_sql(query, conn)
max_distinct_prompts_per_session_id_public_page_or_generated.columns = ['Maximum number of distinct prompts per user session for the public page']

max_distinct_prompts_per_session_id_public_page_or_generated

Time taken to vote in seconds (from submitting the prompt until vote submission)

In [ ]:
query = f"""
	SELECT EXTRACT(EPOCH FROM (b.vote_timestamp - p.timestamp)) AS time_taken_seconds
	FROM battle b JOIN prompt p ON b.prompt_id = p.id
	WHERE b.vote_timestamp IS NOT NULL
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	AND b.vote_timestamp < '{CUTOFF_DATE}'
"""
voting_time_taken_in_seconds = pd.read_sql(query, conn)
voting_time_taken_in_seconds.columns = ['Time taken to vote from submitting prompt including time waiting for response.']

voting_time_taken_in_seconds

In [ ]:
fig = px.histogram(voting_time_taken_in_seconds, title="Time taken to vote from submitting prompt including time waiting for response " + category_and_cutoff_date_figures_text, text_auto=True, height=400, nbins=50)

fig.update_layout(yaxis_title="Votes in time range", xaxis_title="Time in seconds", showlegend=False, font_size=16)

fig

Minimum, maximum, average, and median time taken to vote in seconds

In [ ]:
query = f"""
	SELECT
    MAX(time_taken_seconds) AS max_time,
    MIN(time_taken_seconds) AS min_time,
    AVG(time_taken_seconds) AS avg_time,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY time_taken_seconds) AS median_time
	FROM (
		SELECT
			EXTRACT(EPOCH FROM (b.vote_timestamp - p.timestamp)) AS time_taken_seconds
		FROM battle b
		JOIN prompt p ON b.prompt_id = p.id
		WHERE (b.vote_timestamp IS NOT NULL)
		AND (p.timestamp IS NOT NULL)
		AND (b.vote_timestamp < '{CUTOFF_DATE}')
		AND (p.is_from_public_page = TRUE OR p.session_id = 'generator-session')
		AND (p.category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	) AS time_differences
"""
voting_time_taken_in_seconds_min_max_avg_med = pd.read_sql(query, conn)

voting_time_taken_in_seconds_min_max_avg_med

Lengths of model responses

In [ ]:
query = f"""
	SELECT LENGTH(b.model_1_answer) AS answer_length
	FROM battle b JOIN prompt p ON b.prompt_id = p.id
	WHERE b.model_1_answer IS NOT NULL
	AND p.timestamp < '{CUTOFF_DATE}'
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')

	UNION ALL

	SELECT LENGTH(b.model_2_answer) AS answer_length
	FROM battle b JOIN prompt p ON b.prompt_id = p.id
	WHERE b.model_2_answer IS NOT NULL
	AND p.timestamp < '{CUTOFF_DATE}'
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
"""
answer_lengths = pd.read_sql(query, conn)
answer_lengths.columns = ['Length of responses from AI models']

answer_lengths

In [ ]:
fig = px.histogram(answer_lengths, title="Length of responses from AI models " + category_and_cutoff_date_figures_text, text_auto=True, height=400, nbins=50)
fig.update_layout(yaxis_title="Answers in length range", xaxis_title="Length of response", showlegend=False, font_size=16)
fig

Minimum, maximum, average, and median lengths of model responses

In [ ]:
query = f"""
	SELECT
    MIN(answer_length) AS min_length,
    MAX(answer_length) AS max_length,
    AVG(answer_length) AS avg_length,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY answer_length) AS median_length
	FROM (
		SELECT LENGTH(b1.model_1_answer) AS answer_length
		FROM battle b1 JOIN prompt p1 ON b1.prompt_id = p1.id
		WHERE b1.model_1_answer IS NOT NULL
		AND p1.timestamp < '{CUTOFF_DATE}'
		AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')

		UNION ALL

		SELECT LENGTH(b2.model_2_answer) AS answer_length
		FROM battle b2 JOIN prompt p2 ON b2.prompt_id = p2.id
		WHERE b2.model_2_answer IS NOT NULL
		AND p2.timestamp < '{CUTOFF_DATE}'
		AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	) AS all_lengths
"""
answer_lengths_min_max_avg_med = pd.read_sql(query, conn)
answer_lengths_min_max_avg_med

Lengths of prompts

In [ ]:
query = f"""
	SELECT LENGTH(prompt_text) AS prompt_length
	FROM prompt
	WHERE prompt_text IS NOT NULL
	AND timestamp < '{CUTOFF_DATE}'
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
"""
prompts_lengths = pd.read_sql(query, conn)
prompts_lengths.columns = ['Length of user prompts']

prompts_lengths

In [ ]:
fig = px.histogram(prompts_lengths, title="Length of prompts " + category_and_cutoff_date_figures_text, text_auto=True, height=400, nbins=50)
fig.update_layout(yaxis_title="Prompts in length range", xaxis_title="Length of prompt", showlegend=False, font_size=16)
fig

Minimum, maximum, average, and median lengths of prompts

In [ ]:
query = f"""
	SELECT
    MIN(LENGTH(prompt_text)) AS min_length,
    MAX(LENGTH(prompt_text)) AS max_length,
    AVG(LENGTH(prompt_text)) AS avg_length,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY LENGTH(prompt_text)) AS median_length
	FROM prompt
	WHERE prompt_text IS NOT NULL
	AND (category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	AND timestamp < '{CUTOFF_DATE}'
"""
prompt_lengths_min_max_avg_med = pd.read_sql(query, conn)
prompt_lengths_min_max_avg_med

Win count of the models

In [ ]:
query = f"""
	SELECT m.id, m.model_name, COUNT(m.id) 
	FROM battle b 
		JOIN model m ON b.winner_model_id = m.id 
		JOIN prompt p ON b.prompt_id = p.id
	WHERE b.vote_timestamp < '{CUTOFF_DATE}'
	AND (p.category = '{CATEGORY}' OR '{CATEGORY}' = 'OVERALL')
	GROUP BY m.id,m.model_name
"""
models_win_count = pd.read_sql(query, conn)
models_win_count

Leaderboard score history

In [ ]:
query = f"""
	SELECT *
	FROM(
	SELECT DISTINCT ON (entry_json, category)
		entry_json,
		timestamp
	FROM leaderboard_entry
	WHERE (category = '{CATEGORY}' OR ('{CATEGORY}' = 'OVERALL' AND category IS NULL))
	AND timestamp < '{CUTOFF_DATE}'
	ORDER BY entry_json, category, timestamp ASC 
	) ORDER BY timestamp
"""
score_history = pd.read_sql(query, conn)
score_history

In [ ]:
score_history["entry_json"] = score_history["entry_json"].apply(json.loads)
score_history["scores"] = score_history["entry_json"].apply(lambda lst: {d["id"]: d["score"] for d in lst})
score_history_expanded = pd.concat([score_history[["timestamp"]], score_history["scores"].apply(pd.Series)], axis=1)
score_history_expanded = score_history_expanded.rename(columns=model_id_name_mapping)
score_history_expanded

In [ ]:
# Melt the wide dataframe into long format
score_history_expanded_long = score_history_expanded.melt(id_vars="timestamp", var_name="id", value_name="score")

fig = px.line(
    score_history_expanded_long,
    x="timestamp",
    y="score",
    color="id",
    title="Leaderboard score history per model " + category_and_cutoff_date_figures_text,
    markers=True
)
fig.update_layout(yaxis_title="Score", xaxis_title="Timestamp")
fig.show()

Latest overall scores before cutoff date

In [ ]:
query = f"""
	SELECT *
	FROM leaderboard_entry
	WHERE category IS NULL
	AND timestamp < '{CUTOFF_DATE}'
	ORDER BY timestamp DESC 
	LIMIT 1
"""
latest_scores_overall = pd.read_sql(query, conn)

latest_scores_overall["entry_json"] = latest_scores_overall["entry_json"].apply(json.loads)
latest_scores_overall["scores"] = latest_scores_overall["entry_json"].apply(lambda lst: {d["id"]: d["score"] for d in lst})
latest_scores_overall = pd.concat([latest_scores_overall["scores"].apply(pd.Series)], axis=1)
latest_scores_overall = latest_scores_overall.rename(columns=model_id_name_mapping)
latest_scores_overall

Latest overall ranking

In [ ]:
latest_ranks_overall = latest_scores_overall.copy()
latest_ranks_overall = latest_scores_overall.rank(axis=1, ascending=False, method="dense")
latest_ranks_overall

Latest scores all categories

In [ ]:
query = f"""
	SELECT *
	FROM leaderboard_entry
	WHERE category IS NOT NULL
	AND timestamp < '{CUTOFF_DATE}'
	ORDER BY timestamp DESC 
	LIMIT (
		SELECT COUNT(DISTINCT category)
		FROM leaderboard_entry
		WHERE category IS NOT NULL
	)
"""
latest_scores_all_categories = pd.read_sql(query, conn)


latest_scores_all_categories["entry_json"] = latest_scores_all_categories["entry_json"].apply(json.loads)
latest_scores_all_categories["scores"] = latest_scores_all_categories["entry_json"].apply(lambda lst: {d["id"]: d["score"] for d in lst})
latest_scores_all_categories_expanded = pd.concat([latest_scores_all_categories[["category"]], latest_scores_all_categories["scores"].apply(pd.Series)], axis=1)
latest_scores_all_categories_expanded_renamed = latest_scores_all_categories_expanded.rename(columns=model_id_name_mapping)
latest_scores_all_categories_expanded_renamed

Ranks of the models in all categories

In [ ]:
score_cols = [c for c in latest_scores_all_categories_expanded_renamed.columns if c != "category"]

latest_ranks_all_categories_expanded = latest_scores_all_categories_expanded_renamed.copy()
latest_ranks_all_categories_expanded[score_cols] = latest_scores_all_categories_expanded_renamed[score_cols].rank(axis=1, ascending=False, method="dense")
latest_ranks_all_categories_expanded

Averaging the ranks of all categories to get overall ranking:

In [ ]:
avg_ranks = latest_ranks_all_categories_expanded[score_cols].mean(axis=0).to_frame("avg_rank")
avg_ranks.index.name = "model"
avg_ranks = avg_ranks.sort_values("avg_rank", ascending=True)
avg_ranks

In [ ]:
ranked_avg = avg_ranks.copy()
ranked_avg["final_rank"] = ranked_avg["avg_rank"].rank(
    ascending=True,
    method="dense"
)
ranked_avg = ranked_avg.sort_values("final_rank", ascending=True)
ranked_avg

Averaging the scores of all categories to get overall ranking:

In [ ]:
avg_scores = latest_scores_all_categories_expanded_renamed[score_cols].mean(axis=0).to_frame("avg_score")
avg_scores.index.name = "model"
avg_scores = avg_scores.sort_values("avg_score", ascending=False)
avg_scores

In [ ]:
avg_scores_ranked = avg_scores.copy()
avg_scores_ranked["rank"] = avg_scores_ranked["avg_score"].rank(
    ascending=False,
    method="dense"
)
avg_scores_ranked = avg_scores_ranked.sort_values("rank", ascending=True)
avg_scores_ranked

Weighted averaging scores of all categories to get overall ranking:

Per-model weighting based on battles participated (true participation weighting) - Each model gets a weight per category for averaging depending on how often that model actually competed in a battle that was voted on.

In [ ]:
query = f"""
	SELECT
		m.model_id,
		m.category,
		COUNT(*) AS battle_count
	FROM (
		SELECT 
			b.id AS battle_id,
			b.model1_id AS model_id,
			p.category
		FROM battle b
		JOIN prompt p ON b.prompt_id = p.id
		JOIN model AS model1 ON b.model1_id = model1.id AND model1.active = TRUE
		JOIN model AS model2 ON b.model2_id = model2.id AND model2.active = TRUE
		WHERE (b.vote_timestamp IS NOT NULL)
		AND (b.vote_timestamp < '{CUTOFF_DATE}')

		UNION ALL

		SELECT 
			b.id AS battle_id,
			b.model2_id AS model_id,
			p.category
		FROM battle b
		JOIN prompt p ON b.prompt_id = p.id
		JOIN model AS model1 ON b.model1_id = model1.id AND model1.active = TRUE
		JOIN model AS model2 ON b.model2_id = model2.id AND model2.active = TRUE
		WHERE (b.vote_timestamp IS NOT NULL)
		AND (b.vote_timestamp < '{CUTOFF_DATE}')

	) AS m
	GROUP BY m.model_id, m.category
	ORDER BY m.model_id, m.category
"""
model_voted_battle_counts_per_category = pd.read_sql(query, conn)
model_voted_battle_counts_per_category

In [ ]:
model_voted_battle_counts_per_category_pivot = model_voted_battle_counts_per_category.pivot_table(
    index="category",
    columns="model_id",
    values="battle_count",
    fill_value=0
)

model_voted_battle_counts_per_category_pivot

In [ ]:
normalized_category_weights_per_model = model_voted_battle_counts_per_category_pivot.div(model_voted_battle_counts_per_category_pivot.sum(axis=0), axis=1)
normalized_category_weights_per_model

In [ ]:
latest_scores_all_categories_expanded

In [ ]:
df_scores = latest_scores_all_categories_expanded
df_weights = normalized_category_weights_per_model

#make sure both have the same ordering of categories
if "category" in df_scores.columns:
	df_scores = df_scores.set_index("category")
if "category" in df_weights.columns:
	df_weights = df_weights.set_index("category")

df_scores = df_scores.sort_index()
df_weights = df_weights.sort_index()


#make sure both have the same ordering of columns (models)
df_scores, df_weights = df_scores.align(df_weights, join="inner", axis=1)

In [ ]:
weighted_average_scores = (df_scores * df_weights).sum(axis=0)
weighted_average_scores = weighted_average_scores.rename(index=model_id_name_mapping)
weighted_average_scores.name = "weighted_score"
weighted_average_scores

In [ ]:
weighted_average_scores_and_ranks = weighted_average_scores.to_frame()
weighted_average_scores_and_ranks["rank"] = weighted_average_scores_and_ranks["weighted_score"].rank(ascending=False, method="min").astype(int)
weighted_average_scores_and_ranks = weighted_average_scores_and_ranks.sort_values("rank")
weighted_average_scores_and_ranks

In [ ]:
conn.commit()
conn.close()